# GuindiMa AI — voix wolof CONCREE/Adia_TTS : génération contrôlée

Ce notebook régénère uniquement les WAV marqués `a_regenerer` dans
`scripts/phrases_voix.json`, avec la voix **Adia_TTS** (`CONCREE/Adia_TTS`).

Pour chaque entrée :

- 3 candidats sont générés avec les graines `graine`, `graine + 1`, `graine + 2` ;
- chaque candidat est transcrit par Whisper (fine-tuning Dakar, repli public) ;
- le meilleur est gardé selon la similarité texte et la durée ;
- `rapport_voix.csv` garde les 3 scores, la durée, le candidat retenu et `douteux`.

Le zip final `tts_cache_adia.zip` contient **tous** les WAV présents dans
`tts_cache_adia/` (anciens + nouveaux) et `rapport_voix.csv`.

Sur une session Colab neuve, dépose un zip de l'ancien cache
(`data/tts_cache/` ou un ancien `tts_cache_adia.zip`) et décompresse-le dans
`tts_cache_adia/` avant de lancer : sinon les WAV non régénérés manqueront.
Le notebook refuse de créer un zip incomplet.

GPU obligatoire (T4 suffit). Les phrases restent dans `scripts/phrases_voix.json`.


In [ ]:
# Un GPU doit être attribué, sinon : Runtime > Change runtime type > GPU.
!nvidia-smi

## Installation

1. `parler-tts` (paquet officiel Hugging Face).
2. Colab embarque une version ancienne de `protobuf` (via TensorFlow) ;
   `parler-tts`/`transformers` exigent `protobuf>=5.29`, sinon on tombe sur
   `cannot import name 'runtime_version' from 'google.protobuf'`.

**Après la cellule suivante : redémarre la session**
(`Runtime > Restart session`), puis continue directement à la cellule de
chargement du modèle (ne réinstalle pas).

In [ ]:
!pip install -q git+https://github.com/huggingface/parler-tts.git
!pip install -q "protobuf>=5.29"
print("Installé. Redémarre maintenant la session (Runtime > Restart session).")

In [ ]:
import torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

MODELE = "CONCREE/Adia_TTS"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
assert device == "cuda:0", "Colab doit tourner sur GPU (Runtime > Change runtime type > GPU)"

model = ParlerTTSForConditionalGeneration.from_pretrained(MODELE).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODELE)
print("Chargé :", MODELE, "| fréquence :", model.config.sampling_rate)

In [ ]:
# Récupérer scripts/phrases_voix.json (dépose-le depuis le dépôt local).
import json
import os
from google.colab import files

if not os.path.exists("phrases_voix.json"):
    print("Choisis scripts/phrases_voix.json…")
    envoi = files.upload()
    nom = next(iter(envoi))
    os.rename(nom, "phrases_voix.json")

with open("phrases_voix.json", encoding="utf-8") as f:
    plan = json.load(f)

GRAINE = plan.get("graine", 1234)
DESCRIPTION = plan.get("description", "A warm and natural voice, with a conversational flow")
PHRASES = plan["phrases"]
SORTIE = "tts_cache_adia"
RAPPORT = "rapport_voix.csv"
RAPPORT_CONTROLE = "rapport_voix_controle.csv"
os.makedirs(SORTIE, exist_ok=True)
a_regenerer = [p for p in PHRASES if p.get("a_regenerer")]
print(len(PHRASES), "phrases |", len(a_regenerer), "à régénérer |", DESCRIPTION)


## Vérification automatique

- **Similarité** : normalisation (minuscules, accents, ponctuation), puis
  distance de Levenshtein normalisée entre la phrase et la transcription.
- **Durée** : vitesse de référence de 12 caractères/s, tolérance de 55 %
  (au moins 0,55 s). Un WAV qui dépasse la borne haute est considéré comme
  ayant du son après la fin attendue.
- **Douteux** : meilleur score < 0,5 **ou** durée hors bornes.

Les fonctions de cette cellule sont pures : les tests locaux les exécutent
sans charger Adia ni Whisper.


In [ ]:
# HELPERS_VERIFICATION_VOIX
# Ces fonctions sont pures (ni torch ni numpy) : les tests locaux exécutent
# directement cette cellule pour vérifier le choix du meilleur candidat.
import re
import unicodedata


SEUIL_SIMILARITE = 0.5
VITESSE_CAR_PAR_SEC = 12.0
TOLERANCE_MIN = 0.55
TOLERANCE_RATIO = 0.35
DUREE_MIN = 0.45


def normaliser_texte(texte):
    """Minuscules, sans accents ni ponctuation, pour comparer Whisper au plan."""
    texte = (texte or "").lower()
    texte = texte.translate(str.maketrans({"ŋ": "n", "ë": "e", "ñ": "n"}))
    texte = unicodedata.normalize("NFKD", texte)
    texte = "".join(c for c in texte if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", " ", texte).strip()


def distance_levenshtein(a, b):
    """Distance de caractères classique, en O(min(len(a), len(b))) mémoire."""
    if len(a) < len(b):
        a, b = b, a
    precedente = list(range(len(b) + 1))
    for i, caractere_a in enumerate(a, 1):
        courante = [i]
        for j, caractere_b in enumerate(b, 1):
            cout = 0 if caractere_a == caractere_b else 1
            courante.append(min(
                courante[-1] + 1,
                precedente[j] + 1,
                precedente[j - 1] + cout,
            ))
        precedente = courante
    return precedente[-1]


def similarite(phrase, transcription):
    """Score de 0 à 1 : 1 = transcription identique après normalisation."""
    a = normaliser_texte(phrase)
    b = normaliser_texte(transcription)
    if not a and not b:
        return 1.0
    return 1.0 - distance_levenshtein(a, b) / max(len(a), len(b))


def bornes_duree(phrase):
    """Bornes basses et hautes attendues pour une phrase, en secondes."""
    longueur = len(normaliser_texte(phrase))
    attendu = max(DUREE_MIN, longueur / VITESSE_CAR_PAR_SEC)
    tolerance = max(TOLERANCE_MIN, attendu * TOLERANCE_RATIO)
    return max(DUREE_MIN, attendu - tolerance), attendu + tolerance


def controler_duree(duree_s, phrase):
    """Renvoie (ok, raison) ; la borne haute attrape le son après la fin."""
    mini, maxi = bornes_duree(phrase)
    if duree_s < mini:
        return False, "trop court"
    if duree_s > maxi:
        return False, "trop long / son après la fin attendue"
    return True, ""


def est_douteux(score, duree_ok):
    """Douteux si Whisper doute ou si la durée est hors bornes."""
    return score < SEUIL_SIMILARITE or not duree_ok


def choisir_meilleur_candidat(candidats):
    """Indice du meilleur candidat : similarité, puis durée valide.

    Un candidat trop court ou trop long est pénalisé de moitié : mieux vaut
    une phrase un peu moins bien reconnue qu'un WAV coupé ou prolongé.
    """
    if not candidats:
        raise ValueError("aucun candidat")

    def cle(indice):
        candidat = candidats[indice]
        ajuste = candidat["score"] if candidat.get("duree_ok", True) else candidat["score"] * 0.5
        return ajuste, candidat["score"], -indice

    return max(range(len(candidats)), key=cle)


In [ ]:
# Whisper de contrôle : notre fine-tuning Dakar d'abord, repli public ensuite.
import os

import numpy as np
from transformers import WhisperForConditionalGeneration, WhisperProcessor

WHISPER_PRINCIPAL = "TIJAANI/whisper-small-wolof-dakar-essai3"
WHISPER_SECOURS = "M9and2M/whisper-small-wolof"

# Secret Colab ; hors Colab, on accepte aussi la variable d'environnement.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
HF_TOKEN = (HF_TOKEN or "").strip() or None

if HF_TOKEN is None:
    print("Secret HF_TOKEN absent : repli sur", WHISPER_SECOURS)


def charger_whisper():
    for depot, jeton in ((WHISPER_PRINCIPAL, HF_TOKEN), (WHISPER_SECOURS, None)):
        try:
            processeur = WhisperProcessor.from_pretrained(depot, token=jeton)
            modele = WhisperForConditionalGeneration.from_pretrained(depot, token=jeton).to(device)
            modele.eval()
            print("Whisper de contrôle :", depot)
            return processeur, modele, depot
        except Exception as exc:
            print(f"[Whisper] {depot} inaccessible ({type(exc).__name__} : {exc})")
    raise RuntimeError(f"ni {WHISPER_PRINCIPAL} ni {WHISPER_SECOURS} ne se chargent")


asr_proc, asr_model, asr_depot = charger_whisper()


def reechantillonner_16k(audio, sr):
    """Whisper attend 16 kHz mono ; interpolation linéaire sans dépendance."""
    audio = np.asarray(audio, dtype=np.float32).squeeze()
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr == 16000 or audio.size == 0:
        return audio
    n = int(round(len(audio) * 16000 / sr))
    if n <= 1:
        return audio[:n]
    x_ancien = np.linspace(0.0, 1.0, len(audio), endpoint=False, dtype=np.float64)
    x_nouveau = np.linspace(0.0, 1.0, n, endpoint=False, dtype=np.float64)
    return np.interp(x_nouveau, x_ancien, audio).astype(np.float32)


def transcrire(audio, sr):
    """Transcrit un WAV déjà généré (pas de language : « wo » fait planter Whisper)."""
    audio_16k = reechantillonner_16k(audio, sr)
    entree = asr_proc(audio_16k, sampling_rate=16000, return_tensors="pt")
    with torch.inference_mode():
        ids = asr_model.generate(entree.input_features.to(device), max_new_tokens=96)
    return asr_proc.batch_decode(ids, skip_special_tokens=True)[0].strip()


In [ ]:
# Générer 3 candidats par entrée marquée, garder le meilleur, écrire le rapport.
import csv
import os
import time

import numpy as np
import soundfile as sf

SR = model.config.sampling_rate
GRAINES = [GRAINE, GRAINE + 1, GRAINE + 2]
RAPPORT = "rapport_voix.csv"
CHAMPS_RAPPORT = [
    "fichier", "phrase", "score_1", "score_2", "score_3",
    "duree_s", "retenu", "douteux",
]


def nettoyer(audio, sr, marge=0.05, cible=0.97):
    """Coupe les silences au début/fin, puis égalise le volume."""
    audio = np.asarray(audio, dtype=np.float32).squeeze()
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if audio.size == 0:
        return audio
    seuil = max(1e-3, 0.02 * float(np.max(np.abs(audio))))
    forts = np.where(np.abs(audio) > seuil)[0]
    if forts.size:
        debut = max(0, forts[0] - int(sr * marge))
        fin = min(len(audio), forts[-1] + int(sr * marge))
        audio = audio[debut:fin]
    crete = float(np.max(np.abs(audio))) or 1.0
    return (audio * (cible / crete)).astype(np.float32)


def generer_candidat(phrase, graine):
    torch.manual_seed(graine)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(graine)
    entrees = tokenizer(DESCRIPTION, return_tensors="pt").input_ids.to(device)
    invite = tokenizer(phrase, return_tensors="pt").input_ids.to(device)
    with torch.inference_mode():
        audio = model.generate(input_ids=entrees, prompt_input_ids=invite)
    return nettoyer(audio.float().cpu().numpy(), SR)


def ecrire_rapport(lignes, chemin=RAPPORT):
    with open(chemin, "w", encoding="utf-8-sig", newline="") as fichier:
        writer = csv.DictWriter(fichier, fieldnames=CHAMPS_RAPPORT)
        writer.writeheader()
        writer.writerows(lignes)


a_regenerer = [p for p in PHRASES if p.get("a_regenerer")]
lignes_rapport = []
douteux = []
debut = time.perf_counter()

for n, item in enumerate(a_regenerer, 1):
    candidats = []
    for numero, graine in enumerate(GRAINES, 1):
        audio = generer_candidat(item["phrase"], graine)
        duree = len(audio) / SR
        duree_ok, raison = controler_duree(duree, item["phrase"])
        try:
            transcription = transcrire(audio, SR)
        except Exception as exc:
            transcription = ""
            print(f"  transcription impossible ({type(exc).__name__} : {exc})")
        score = similarite(item["phrase"], transcription)
        candidats.append({
            "graine": graine,
            "audio": audio,
            "duree": duree,
            "score": score,
            "duree_ok": duree_ok,
            "raison": raison,
        })
        print(
            f"  candidat {numero}/{len(GRAINES)} : score={score:.3f} "
            f"durée={duree:.2f}s" + ("" if duree_ok else f" ({raison})")
        )

    meilleur = choisir_meilleur_candidat(candidats)
    retenu = candidats[meilleur]
    chemin = os.path.join(SORTIE, item["fichier"])
    sf.write(chemin, retenu["audio"], SR, subtype="PCM_16")

    est_douteux_item = est_douteux(retenu["score"], retenu["duree_ok"])
    ligne = {
        "fichier": item["fichier"],
        "phrase": item["phrase"],
        "score_1": round(candidats[0]["score"], 4),
        "score_2": round(candidats[1]["score"], 4),
        "score_3": round(candidats[2]["score"], 4),
        "duree_s": round(retenu["duree"], 3),
        "retenu": f"{meilleur + 1} (graine {retenu['graine']})",
        "douteux": "oui" if est_douteux_item else "non",
    }
    lignes_rapport.append(ligne)
    ecrire_rapport(lignes_rapport)
    if est_douteux_item:
        douteux.append(item["fichier"])
    print(
        f"[{n}/{len(a_regenerer)}] {item['fichier']} : "
        f"score={retenu['score']:.3f} durée={retenu['duree']:.2f}s "
        f"douteux={'oui' if est_douteux_item else 'non'}"
    )

if a_regenerer:
    ecrire_rapport(lignes_rapport)
    minutes = (time.perf_counter() - debut) / 60
    print(f"\n{len(a_regenerer)} entrées, {len(a_regenerer) * 3} candidats en {minutes:.1f} min")
    print("Rapport :", RAPPORT)
    print("Douteux :", douteux or "aucun")
else:
    print("Aucune entrée a_regenerer=True : rien à générer.")


## Contrôle seul (facultatif)

À lancer après la génération, ou seul sur un cache existant : transcrit tous
les WAV présents **sans régénérer**, applique les mêmes seuils et écrit
`rapport_voix_controle.csv`. Les doutes sont aussi listés à l'écran.


In [ ]:
# Contrôle seul : aucun appel à Adia, aucun WAV modifié.
import csv
import os

import soundfile as sf

RAPPORT_CONTROLE = "rapport_voix_controle.csv"
CHAMPS_CONTROLE = ["fichier", "phrase", "score", "duree_s", "douteux", "raison"]
lignes_controle = []
douteux_controle = []

for item in PHRASES:
    chemin = os.path.join(SORTIE, item["fichier"])
    if not os.path.exists(chemin):
        continue
    audio, sr = sf.read(chemin, dtype="float32", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    duree = len(audio) / sr
    duree_ok, raison = controler_duree(duree, item["phrase"])
    try:
        transcription = transcrire(audio, sr)
    except Exception as exc:
        transcription = ""
        raison = f"transcription impossible ({type(exc).__name__})"
    score = similarite(item["phrase"], transcription)
    est_douteux_item = est_douteux(score, duree_ok)
    if est_douteux_item:
        if not raison:
            raison = "similarité faible"
        douteux_controle.append(item["fichier"])
    lignes_controle.append({
        "fichier": item["fichier"],
        "phrase": item["phrase"],
        "score": round(score, 4),
        "duree_s": round(duree, 3),
        "douteux": "oui" if est_douteux_item else "non",
        "raison": raison,
    })

with open(RAPPORT_CONTROLE, "w", encoding="utf-8-sig", newline="") as fichier:
    writer = csv.DictWriter(fichier, fieldnames=CHAMPS_CONTROLE)
    writer.writeheader()
    writer.writerows(lignes_controle)

print(f"{len(lignes_controle)} WAV contrôlés, {len(douteux_controle)} douteux")
print("Rapport :", RAPPORT_CONTROLE)
print("Douteux :", douteux_controle or "aucun")


In [ ]:
# Écoute rapide d'un exemple régénéré avant de tout empaqueter.
import os

from IPython.display import Audio, display

exemple = next(
    (
        p for p in PHRASES
        if p.get("a_regenerer") and os.path.exists(os.path.join(SORTIE, p["fichier"]))
    ),
    PHRASES[0],
)
print(exemple["phrase"])
display(Audio(os.path.join(SORTIE, exemple["fichier"])))


In [ ]:
# Empaqueter tous les WAV (anciens + nouveaux) et le rapport, puis télécharger.
import os
import shutil

from google.colab import files

if not os.path.exists(RAPPORT):
    raise RuntimeError("rapport_voix.csv absent : lance d'abord la génération.")

manquants = [
    p["fichier"] for p in PHRASES
    if not os.path.exists(os.path.join(SORTIE, p["fichier"]))
]
if manquants:
    raise RuntimeError(
        f"{len(manquants)} WAV manquants dans {SORTIE}/ (ex. {manquants[:5]}). "
        "Dépose un zip de l'ancien cache (data/tts_cache/) et décompresse-le ici."
    )

shutil.copy(RAPPORT, os.path.join(SORTIE, "rapport_voix.csv"))
if os.path.exists(RAPPORT_CONTROLE):
    shutil.copy(RAPPORT_CONTROLE, os.path.join(SORTIE, RAPPORT_CONTROLE))

shutil.make_archive("tts_cache_adia", "zip", SORTIE)
files.download("tts_cache_adia.zip")
